In [ ]:
import glob
import json

import matplotlib.pyplot as plt
import numpy as np
import tifffile

from natsort import natsorted
from sklearn.linear_model import LinearRegression

%matplotlib inline
plt.style.use('seaborn-v0_8-notebook')

# A session recorded with scripts/basler_live.py, e.g. by running it with --open data/live/practical.
# To try the notebook without the camera: .venv/bin/python scripts/simulate_session.py data/live/practical
data_dir = 'data/live/practical/'

def load_tiff_directory(directory_path: str, downsample: int = 1):
    """Every frame in a recording, keeping every `downsample`th pixel in each direction."""
    tiff_paths = natsorted(glob.glob(directory_path + '*.tif*'))
    curr_img = tifffile.imread(tiff_paths[0])[::downsample, ::downsample]
    images = np.empty((len(tiff_paths), curr_img.shape[0], curr_img.shape[1]), dtype=np.float32)
    images[0] = curr_img

    for i, tiff_path in enumerate(tiff_paths[1:]):
        images[i+1] = tifffile.imread(tiff_path)[::downsample, ::downsample]
        
    return images

def load_metadata(directory_path: str):
    """The settings read back from the camera when the recording was made."""
    with open(directory_path + 'metadata.json') as f:
        return json.load(f)

**A note on units:** the recordings are the camera's raw 12-bit values (Mono12, 0-4095), stored in 16-bit TIFFs without any shift. All counts below are in these 12-bit units.

Every recording folder also holds `metadata.json`, with the gain, exposure time and black level read back from the camera, so nothing has to be inferred from folder names.

## Measure readout noise

### Data: 
A sequence of images (∼ 100) from the camera with NO light hitting the detector at an extremely short exposure time (∼ 1 ms). Recorded with the Read noise tab of `basler_live.py`, at the camera's minimum exposure.

Set the black level (bias) high enough that no pixel reads 0 before recording. Otherwise the negative fluctuations are clipped at 0 and the standard deviation comes out too low.

### Exercise instructions:
Measure the standard deviation of each pixels value across the movie (use ImageJ). What is the average noise (standard deviation in counts) across pixels?

In [ ]:
def steady_frames(frames, max_step=0.5):
    """The longest run of frames whose mean doesn't jump (the later run if two are as long).

    The camera's offset sometimes steps by about 2 counts during a stack. Averaging across
    the step would mix two biases.
    """
    means = frames.mean(axis=(1, 2), dtype=np.float64)
    breaks = np.flatnonzero(np.abs(np.diff(means)) > max_step) + 1
    runs = np.split(np.arange(len(means)), breaks)
    longest = max(reversed(runs), key=len)
    return frames[longest[0]:longest[-1] + 1]

def measure_readout_noise(im_data, downsample=5):
    im_data = im_data[:, ::downsample, ::downsample]
    
    return np.std(im_data, axis=0, ddof=1, dtype=np.float32)

def calc_plot_readout_noise(im_data, axis, downsample=5):
    pix_dev = measure_readout_noise(im_data, downsample)
    mean_dev = np.mean(pix_dev, dtype=np.float32)
    
    axis.hist(pix_dev.flatten(), bins=100)
    axis.axvline(mean_dev, color='r', linestyle='--')
    axis.text(0.8, 0.8, f'Mean: {mean_dev:.3f}', ha='center', va='center', transform=axis.transAxes)
    axis.set_ylabel('Count')
    axis.set_xlabel('Standard deviation')
    
    return mean_dev

def plot_readout_noise_multiple_gain(gain_levels: list[str], path_stub: str,  downsample: int):
    num_plots = (len(gain_levels) + 1)// 2
    
    fig, ax = plt.subplots(num_plots, 2, figsize=(14, 14))
    mean_devs = []
    bias = []
    
    for i, gain in enumerate(gain_levels):
        read_noise_images = steady_frames(load_tiff_directory(f'{path_stub}{gain}dB/'))
        bias.append(np.mean(read_noise_images, dtype=np.float32))
        print(f'{gain} dB: {np.mean(read_noise_images == 0):.1%} of values are clipped at 0')
        mean_devs.append(calc_plot_readout_noise(read_noise_images, ax[i//2][i%2], downsample))
        ax[i//2][i%2].set_title(f'{gain}dB')
        
    plt.show()
    
    fig, ax = plt.subplots()
    ax.plot(gain_levels, mean_devs, marker='o')
    ax.set_xlabel('Gain (dB)')
    ax.set_ylabel('Read noise (counts)')
    plt.show()
    
    return mean_devs, bias

Let's plot the readout noise for the Basler camera at different gain levels.

In [ ]:
gain_levels = [0, 1, 12, 24]
read_noise_stub = f'{data_dir}read_noise/'
downsample_by = 5

# Check which camera recorded the session: the simulator's recordings say "Simulated"
print(f"Camera: {load_metadata(f'{read_noise_stub}{gain_levels[0]}dB/')['camera_model']}")
read_noise, read_bias = plot_readout_noise_multiple_gain(gain_levels, read_noise_stub, downsample_by)

### Conclusions

*Fill in the numbers after the live session.*

**Check the printout first: 0% of values should be clipped at 0.** If any are, the black level was too low, so the negative fluctuations were cut off and the standard deviation is too low. Record again with a higher black level.

| Gain | Bias (counts) | Read noise (counts) |
|---|---|---|
| 0 dB | | |
| 1 dB | | |
| 12 dB | | |
| 24 dB | | |

Read noise in counts increases with gain because the amplifier also amplifies noise generated before it.

## Measure dark counts

### Data:
A sequence of images (∼ 100) from the camera with NO light hitting the detector at a range of exposure times.

### Exercise instructions:
Measure the Mean number of counts in each image. 

Do the counts increase with increasing exposure time? How many counts/pixel/second do you observe? Hint: Remember to subtract the Bias Level, the average value (in counts) appearing in the image following a 0-second exposure. (This is an offset added to each image by the camera manufacturers and does not reflect the incident light level.)

The Dark counts tab of `basler_live.py` records a short stack at each exposure, for each gain, into `dark/<gain>dB/<exposure>us/`.

In [ ]:
def measure_dark_counts(im_data):
    return np.mean(im_data, axis=(1,2), dtype=np.float32)

def calc_plot_dark_counts(im_data, exposures):
    dark_counts = measure_dark_counts(im_data)
    bias = dark_counts[0]
    subtracted_counts = dark_counts - bias
    
    model_dark_noise = LinearRegression()
    model_dark_noise.fit(exposures.reshape(-1, 1), subtracted_counts)
    
    print(f'The slope of the dark noise is {model_dark_noise.coef_}')
    print(f'The y-intercept of the dark noise is {model_dark_noise.intercept_}')
    
    fig, ax = plt.subplots()
    ax.plot(exposures, subtracted_counts, marker='o')
    ax.set_xlabel('Exposure time (s)')
    ax.set_ylabel('Mean counts')
    plt.show()
    
    return dark_counts, subtracted_counts, model_dark_noise

def load_dark_series(gain_dir):
    """The mean image at each exposure, and the exposures in seconds from metadata.json."""
    folders = sorted(glob.glob(gain_dir + '*us/'), key=lambda f: load_metadata(f)['exposure_us'])
    exposures = np.array([load_metadata(f)['exposure_us'] * 1e-6 for f in folders])
    images = np.stack([steady_frames(load_tiff_directory(f)).mean(axis=0) for f in folders])
    return images, exposures

In [ ]:
dark_count_results = []
dark_exposures = []

for gain in gain_levels:
    print(f'Dark counts for {gain}dB')
    dark_images, exposures = load_dark_series(f'{data_dir}dark/{gain}dB/')
    dark_exposures.append(exposures)
    dark_count_results.append(calc_plot_dark_counts(dark_images, exposures))

### Conclusions

*Fill in the numbers after the live session.*

| Gain | Bias (counts) | Dark counts (counts/pixel/s) | Dark counts (e-/pixel/s) |
|---|---|---|---|
| 0 dB | | | |
| 1 dB | | | |
| 12 dB | | | |
| 24 dB | | | |

The bias is the mean at the shortest exposure, where dark counts are negligible. The fit intercepts should be close to 0 after subtracting it. Dark counts rise with gain because electrons are amplified whatever their source. Convert them to e-/pixel/s with the gain measured in the shot noise section.

## Compute the dynamic range of the camera

First determine the cameras full well capacity (saturation value) in counts after bias removal.

Use a long exposure and expose the camera to a bright light source. What value are saturated
pixels assigned? Use the formula below to compute the cameras dynamic range (the number of
individually discernible light levels) for a fixed exposure.
$$
\text{Dynamic Range} = (\text{Saturation} - \text{Dark counts}) / \text{Read-out noise}
$$
Note: Use bias subtracted values for both Saturation and Dark counts.
Compare your results to the cameras technical specification sheet.

In [ ]:
# metadata.json records the largest value the camera can output: 4095 for 12-bit Mono12.
saturated_value = load_metadata(f'{read_noise_stub}{gain_levels[0]}dB/')['max_value']
print(f'Saturated pixels are assigned the value {saturated_value}')

# A recording from the Dynamic range tab's "Record saturated frames" confirms it
for path in natsorted(glob.glob(f'{data_dir}saturation/*/')):
    print(f'{path}: brightest value {int(load_tiff_directory(path).max())}')

exposure_index = 1  # 1 ms in the default exposure list
dynamic_range = []

for i, gain in enumerate(gain_levels):
    # dark_count_results[i][0][0] is the bias for the given gain
    bias = dark_count_results[i][0][0]
    sat_value = saturated_value - bias
    dark_count = dark_count_results[i][1][exposure_index]
    dynamic_range.append((sat_value - dark_count) / read_noise[i])

    print(f'{gain} dB: saturation after bias removal is {sat_value:.0f} counts, '
          f'dark counts at {dark_exposures[i][exposure_index]*1000:.2f} ms are {dark_count:.2f}, '
          f'dynamic range is {dynamic_range[i]:.0f}:1 ({20 * np.log10(dynamic_range[i]):.1f} dB)')

### Conclusions

*Fill in the numbers after the live session.*

| Gain | Saturation after bias removal (counts) | Dynamic range | Dynamic range (dB) |
|---|---|---|---|
| 0 dB | | | |
| 1 dB | | | |
| 12 dB | | | |
| 24 dB | | | |

Expect the dynamic range to be highest at the lowest gain: increasing the gain raises read noise in counts, while the digitizer's maximum value stays fixed. The comparison with the specification sheet is at the end of the notebook.

## Measure shot noise

### Data:
A sequence of 100 images at each gain, with the camera lit unevenly so that the image runs from dark to saturated. Each pixel sees a different brightness, so one recording gives the mean and variance over the whole range of brightness.

Record with the Shot noise tab of `basler_live.py`, which saves `shot/<gain>dB/`. It scales the exposure with gain so the image stays about as bright at every gain. The light must be stable; use an LED in an enclosed container (to avoid background changes).

### Exercise instructions:
Plot the variance, i.e. standard deviation squared (Y axis) vs. mean (X axis) for each pixel. Fit a line. Determine the detectors gain (counts per photoelectron) at each gain setting.

Hint: Remember to subtract out the Bias Level measured above, at the same gain. Leave out pixels that read 0 or saturate in any frame: they are clipped, so their variance is too low. Pixels close to saturation also have too little variance, because their wells are filling up: find the mean where the variance peaks, and fit only below about 70% of it.

Note: The slope of the line will be the Gain of your camera.

Where does the fit line intersect the y-axis? Does this make sense?

In [ ]:
def check_frames(directory_path):
    """The camera numbers its frames, and basler_live.py records any it missed or repeated."""
    meta = load_metadata(directory_path)
    assert meta['missing_frames'] == 0 and meta['repeated_frames'] == 0, f'{directory_path}: frames were dropped or repeated'
    return meta

def measure_shot_noise(image_path, bias, saturated_value, downsample=4):
    check_frames(image_path)
    shot_image = load_tiff_directory(image_path, downsample)
    shot_image = shot_image.reshape(shot_image.shape[0], -1)

    # A pixel that reads 0 or saturates in any frame is clipped, so its variance is too low
    saturated = shot_image.max(axis=0) >= saturated_value
    usable = (shot_image.min(axis=0) > 0) & ~saturated
    print(f'{usable.mean():.1%} of pixels are usable, {saturated.mean():.1%} saturate')
    shot_image = shot_image[:, usable] - bias

    # Rescale each frame to the same mean so that frame-to-frame changes in the
    # illumination are not counted as shot noise.
    frame_means = np.mean(shot_image, axis=1, dtype=np.float64)
    print(f'{shot_image.shape[0]} frames, frame means vary by {np.std(frame_means, ddof=1) / np.mean(frame_means):.2%}')
    shot_image = shot_image / (frame_means / np.mean(frame_means)).reshape(-1, 1).astype(np.float32)

    mean = np.mean(shot_image, axis=0, dtype=np.float64).reshape(-1, 1)
    var = np.var(shot_image, axis=0, ddof=1, dtype=np.float64).reshape(-1, 1)

    return mean, var

def calculate_plot_shot_noise(image_path, bias, saturated_value, axis, downsample=4):
    mean, var = measure_shot_noise(image_path, bias, saturated_value, downsample)

    # A pixel whose well is nearly full stops gaining variance before it reaches the largest
    # value. Find the mean where the variance peaks (saturation), averaging pixels in bins of
    # similar mean, and fit only below 70% of it, as in the EMVA 1288 standard.
    edges = np.linspace(mean.min(), mean.max(), 51)
    which = np.clip(np.digitize(mean[:, 0], edges) - 1, 0, 49)
    pixels_per_bin = np.bincount(which, minlength=50)
    filled = pixels_per_bin >= 10
    bin_mean = np.bincount(which, weights=mean[:, 0], minlength=50)[filled] / pixels_per_bin[filled]
    bin_var = np.bincount(which, weights=var[:, 0], minlength=50)[filled] / pixels_per_bin[filled]
    saturation_mean = bin_mean[np.argmax(bin_var)] if filled.any() else mean.max()
    fit_pixels = mean[:, 0] < 0.7 * saturation_mean
    print(f'The variance peaks at a mean of {saturation_mean:.0f} counts; fitting {fit_pixels.sum()} pixels below {0.7 * saturation_mean:.0f}')

    model = LinearRegression()
    model.fit(mean[fit_pixels], var[fit_pixels])

    # The intercept is an extrapolation, so check how much it depends on the fit range
    dim_pixels = mean[:, 0] < 0.35 * saturation_mean
    model_dim = LinearRegression()
    model_dim.fit(mean[dim_pixels], var[dim_pixels])

    print(f'The gain of the camera is {model.coef_[0][0]:.3f} counts/e-')
    print(f'The y-intercept is {model.intercept_[0]:.3f}')
    print(f'Fitting the dimmest pixels only, the gain is {model_dim.coef_[0][0]:.3f} counts/e- and the y-intercept is {model_dim.intercept_[0]:.3f}')
    print(f'The dimmest pixel has a mean of {mean.min():.0f} counts\n')

    every = max(1, mean.shape[0] // 20000)
    axis.scatter(mean[::every], var[::every], s=2, alpha=0.3)
    axis.plot(bin_mean, bin_var, 'ko', markersize=4)
    xs = np.array([[0], [0.7 * saturation_mean]])
    axis.plot(xs, model.predict(xs), color='r', linestyle='--')
    axis.axvline(0.7 * saturation_mean, color='grey', linestyle=':')
    axis.set_xlabel('Mean counts - bias')
    axis.set_ylabel('Variance (counts$^2$)')

    return model.coef_[0][0], model.intercept_[0]

In [ ]:
# One unevenly lit recording per gain, shot/<gain>dB/. The bias comes from the read noise
# recording at the same gain, so both must be at the same black level.
num_plots = (len(gain_levels) + 1) // 2
fig, ax = plt.subplots(num_plots, 2, figsize=(14, 6 * num_plots))
shot_results = []

for i, gain in enumerate(gain_levels):
    path = f'{data_dir}shot/{gain}dB/'
    meta = load_metadata(path)
    assert meta['black_level'] == load_metadata(f'{read_noise_stub}{gain}dB/')['black_level'], 'record read noise at the same black level'
    print(f"Shot noise at {meta['gain']:g} dB, {meta['exposure_us'] / 1000:g} ms exposure")
    gain_counts_per_e, intercept = calculate_plot_shot_noise(path, read_bias[i], saturated_value, ax[i//2][i%2])
    ax[i//2][i%2].set_title(f'{gain} dB')
    shot_results.append({'gain_db': meta['gain'], 'counts_per_e': gain_counts_per_e, 'intercept': intercept})

plt.show()

In [ ]:
camera_gain = np.array([r['counts_per_e'] for r in shot_results])

for i, gain in enumerate(gain_levels):
    dark_rate = dark_count_results[i][2].coef_[0]
    implied = np.sqrt(shot_results[i]['intercept']) if shot_results[i]['intercept'] > 0 else np.nan
    print(f"{gain} dB: gain {camera_gain[i]:.3f} counts/e- ({1 / camera_gain[i]:.2f} e- per count), "
          f"read noise {read_noise[i] / camera_gain[i]:.2f} e-, dark counts {dark_rate / camera_gain[i]:.2f} e-/pixel/s")
    print(f"    read noise implied by the y-intercept is {implied:.2f} counts, compared to {read_noise[i]:.2f} counts measured directly")

In [ ]:
# The measured gain should follow the nominal 10^(dB/20), relative to the lowest gain
fig, ax = plt.subplots()
ax.semilogy(gain_levels, camera_gain, marker='o', linestyle='none', label='measured')
ax.semilogy(gain_levels, camera_gain[0] * 10 ** ((np.array(gain_levels) - gain_levels[0]) / 20), linestyle='--', label='nominal')
ax.set_xlabel('Gain (dB)')
ax.set_ylabel('Gain (counts/e-)')
ax.legend()
plt.show()

for i, gain in enumerate(gain_levels):
    print(f"{gain} dB: {camera_gain[i]:.3f} counts/e-, "
          f"implying {camera_gain[i] / 10 ** ((gain - gain_levels[0]) / 20):.3f} counts/e- at {gain_levels[0]} dB")

In [ ]:
# Measured directly at the lowest gain
gain_0db = camera_gain[0]
sat_0db = saturated_value - dark_count_results[0][0][0]
print(f'At {gain_levels[0]} dB: gain {gain_0db:.3f} counts/e-, read noise {read_noise[0] / gain_0db:.2f} e-, '
      f'saturation {sat_0db / gain_0db:.0f} e-, dynamic range {dynamic_range[0]:.0f}:1 ({20 * np.log10(dynamic_range[0]):.1f} dB)')

## Conclusions

*Fill in the numbers after the live session.*

### Shot noise

The slope of variance against mean is the gain in counts/e-. Check the printout for three things:
- **Usable pixels:** pixels that read 0 or saturate in any frame are left out, and the fit stops at 70% of the mean where the variance peaks. If few pixels are fitted, the image was too bright.
- **Frame means:** if they vary by more than a fraction of a percent, the light was not stable. The frames are rescaled to a common mean so that the extra variance does not inflate the slope.
- **Dropped frames:** `check_frames` stops if the camera reported missing or repeated frames, because those are not independent measurements.

| Gain setting | Gain (counts/e-) | e- per count | Read noise (e-) | Dark counts (e-/pixel/s) |
|---|---|---|---|---|
| 0 dB | | | | |
| 1 dB | | | | |
| 12 dB | | | | |
| 24 dB | | | | |

The measured gain should follow the nominal gain setting, rising by a factor of 10 every 20 dB. Read noise in electrons falls as the gain rises, because less of it is added after the amplifier.

### Where does the fit line intersect the y-axis? Does this make sense?

Theory predicts `var(c) = αE[c] + var(read noise)`, so the intercept should be the read noise variance measured directly in the first section. It is an extrapolation: the further the dimmest pixel is from 0, and the more the relationship curves, the less precise it is. Dark counts collected during the exposure also add a little to both the mean and the variance, because the bias was measured at the minimum exposure. Compare the all-pixel and dimmest-pixel fits.

| Gain setting | Fit | Intercept (counts²) | Implied read noise (counts) | Read noise measured directly (counts) |
|---|---|---|---|---|
| | All pixels | | | |
| | Dimmest pixels only | | | |

### Comparison with the specification sheet

The specification values describe the camera at its base gain, so compare them to the 0 dB results, where the gain was measured directly.

| Quantity at 0 dB | Measured | Specification |
|---|---|---|
| Read noise | | 2 e- |
| Saturation | | 10.2 ke- |
| Dynamic range | | 73 dB |

The specification values are [Basler's EMVA 1288 figures](https://assets-ctf.baslerweb.com/dg51pdwahxgw/1hcLdj4BXRUzhnOL2gaNIm/faa0fa7fe5ea641aca3c3be7efe82c24/BAS2606_EMVA_Data_Overview_A4_EN_SAP0136_No_18.pdf) for the IMX304, the sensor in the acA4112-20um.

Saturation in electrons only reflects the true full well capacity at low gain. At higher gain the digitizer reaches its maximum first.